# Data Quality — shopee

Jalankan **Run All** dengan kernel `env`. Keenam pemeriksaan di bawah memakai aturan yang sama untuk semua sumber. Hasil transaksi mengikuti kontrak canonical 12 kolom; `product_id` memakai SKU asli dari Product Master.

Product Master tetap berisi identitas produk dan harga referensi. Data sumber tetap utuh. Semua aturan dijalankan saat persiapan agar pemeriksaan duplikat sudah memperhitungkan hasil mapping produk; enam bagian berikut memperlihatkan hasil tiap pemeriksaan.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pipeline/validation/analysis.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline.validation.analysis import (
    load_analysis, standard_data, missing_values, quality_issues,
    type_report, summary, export_analysis,
)

SOURCE = "shopee"
hasil = load_analysis(SOURCE, ROOT / "data/source")
data_bersih = standard_data(hasil)


## 1. Missing value

Field wajib: order ID, tanggal, produk, quantity, harga satuan, status; total transaksi Website juga wajib. Semua field master wajib. Baris yang tidak memenuhi syarat ditolak. Customer/kota/email yang tidak tersedia tetap kosong, tanpa dummy. Field opsional kosong yang tersedia dalam source dicatat sebagai warning.

In [2]:
display(missing_values(hasil))
display(quality_issues(hasil, "missing"))

,sumber,kolom,jumlah_kosong
0,shopee,order_id,0
1,shopee,order_date,0
2,shopee,product_name,0
3,shopee,qty,0
4,shopee,unit_price,0
5,shopee,customer_name,1
6,shopee,customer_city,1
7,shopee,payment_method,1
8,shopee,status,0


,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,shopee,12,WARNING,customer_name,MISSING_OPTIONAL,
1,shopee,16,WARNING,customer_city,MISSING_OPTIONAL,
2,shopee,20,WARNING,payment_method,MISSING_OPTIONAL,


## 2. Duplicate

Business key: `(channel, order_id)` untuk dataset satu item per order saat ini; master memakai `product_id`/SKU. Record yang sama disimpan satu kali. Jika key sama tetapi nilainya berbeda, semua versi ditolak untuk ditinjau. Bila kelak order memiliki beberapa item, tambahkan line ID dari sumber.

In [3]:
display(quality_issues(hasil, "duplicate"))

,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,shopee,51,INFO,business_key,DUPLICATE_BUSINESS_KEY,SHP-0006


## 3. Invalid value

Quantity wajib integer positif. Harga wajib positif dan maksimal dua desimal. Total harus sama dengan quantity × harga satuan. Status dataset saat ini: `Completed`, `Cancelled`, `Returned`; status lain ditolak. Harga berbeda dari master diberi warning karena mungkin promo. Total harga adalah nilai bruto; hitung penjualan selesai hanya dari `Completed`.

In [4]:
display(quality_issues(hasil, "invalid"))

,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,shopee,4,ERROR,qty,NON_POSITIVE,0
1,shopee,8,ERROR,unit_price,NON_POSITIVE,-50000
2,shopee,24,ERROR,status,INVALID_STATUS,in_progress


## 4. Date format

Tanggal Shopee/Tokopedia: DD/MM/YYYY; Website: MMM DD, YYYY; Offline: DD-MMM-YYYY. Hasil CSV selalu YYYY-MM-DD. Tanggal tidak valid ditolak. Product Master tidak memiliki tanggal transaksi.

In [5]:
display(quality_issues(hasil, "date"))
if "tanggal_order" in data_bersih:
    display(data_bersih[["order_id", "tanggal_order"]].head(5))
else:
    print("Tidak berlaku: master produk tidak memiliki tanggal transaksi.")

,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,shopee,32,ERROR,order_date,INVALID_DATE,not-a-date


,order_id,tanggal_order
0,SHP-0001,2026-08-20
1,SHP-0002,2026-01-14
2,SHP-0003,2026-06-27
3,SHP-0005,2026-02-24
4,SHP-0006,2026-05-04


## 5. Data type

Identifier string, quantity Int64, tanggal datetime, dan uang Decimal. CSV tidak menyimpan tipe data; ekspor tanggal menggunakan YYYY-MM-DD. Teks dirapikan spasinya tanpa merusak nama brand, SKU, shade, atau SPF/PA++++.

In [6]:
display(type_report(data_bersih))

,kolom,dtype,tipe_nilai
0,order_id,string,str
1,product_id,string,str
2,product_name,string,str
3,kategori,string,str
4,quantity,Int64,int64
5,total_harga,object,Decimal
6,tanggal_order,datetime64[us],Timestamp
7,kota,string,str
8,channel,string,str
9,status,string,str


## 6. Product consistency

Variasi huruf besar/kecil, spasi, underscore, dan hyphen dicocokkan ke master. Nama produk dan kategori mengikuti master. Produk tidak dikenal/typo ambigu ditolak, tanpa menebak SKU. Pada master, SKU harus unik.

In [7]:
display(data_bersih[["product_id", "product_name", "kategori"]].drop_duplicates())
display(quality_issues(hasil, "product"))

,product_id,product_name,kategori
0,WRD-MUP-002,Wardah Colorfit Perfect Glow Cushion 13N,Makeup
1,LBR-SKC-002,LABORE Barrier Revive Cream 30ml,Skincare
2,EMN-SKC-001,Emina Bright Stuff Face Wash 100ml,Skincare
3,KHF-BDY-001,Kahf Face Wash Oil and Comedo Defense 100ml,Mens Grooming
5,EMN-SKC-002,Emina Sun Battle SPF 50 PA++++ 30ml,Skincare
6,WND-BDY-001,Wonderly Body Mist Sweet Blossom 100ml,Fragrance
7,MKO-MUP-002,Make Over Powerstay Matte Powder Foundation N20,Makeup
8,BDF-BDY-001,Biodef Body Wash Fresh Care 450ml,Body Care
9,WRD-MUP-001,Wardah Colorfit Velvet Matte Lip Mousse 03,Makeup
10,CRY-SKC-001,Crystallure Supreme Revitalizing Oil Serum 20ml,Skincare


,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,shopee,28,ERROR,product_name,UNMAPPED_PRODUCT,Brigthning Serumm 30ml


## Hasil akhir

Format dan urutan kolom sama untuk semua transaksi. `kota` adalah kota pelanggan online atau kota toko offline; Website yang tidak memiliki kota tetap kosong. Nama channel tetap Shopee/Tokopedia/Website/Offline Store agar bisa dibandingkan.

Hasil utama: `data/processed/clean/`. Notebook ini menyimpan file bersih sumber yang dibahas; `analisa.ipynb` menyimpan seluruh sumber, `sales.csv`, `summary.csv`, serta satu `quality_issues.csv` untuk detail masalah.

In [8]:
ringkasan = summary(hasil)
assert (ringkasan["awal"] == ringkasan["bersih"] + ringkasan["duplikat"] + ringkasan["ditolak"]).all()
display(ringkasan)
display(data_bersih.head(10))
folder_hasil = export_analysis(hasil)
print("Tersimpan:", folder_hasil)

,sumber,awal,bersih,duplikat,ditolak
0,shopee,51,45,1,5


,order_id,product_id,product_name,kategori,quantity,total_harga,tanggal_order,kota,channel,status,customer_email,harga_satuan
0,SHP-0001,WRD-MUP-002,Wardah Colorfit Perfect Glow Cushion 13N,Makeup,1,109000.00,2026-08-20,Semarang,Shopee,Cancelled,<NA>,109000.00
1,SHP-0002,LBR-SKC-002,LABORE Barrier Revive Cream 30ml,Skincare,3,357000.00,2026-01-14,Surabaya,Shopee,Returned,<NA>,119000.00
2,SHP-0003,EMN-SKC-001,Emina Bright Stuff Face Wash 100ml,Skincare,3,83700.00,2026-06-27,Bandung,Shopee,Completed,<NA>,27900.00
3,SHP-0005,KHF-BDY-001,Kahf Face Wash Oil and Comedo Defense 100ml,Mens Grooming,1,42900.00,2026-02-24,Semarang,Shopee,Completed,<NA>,42900.00
4,SHP-0006,KHF-BDY-001,Kahf Face Wash Oil and Comedo Defense 100ml,Mens Grooming,2,85800.00,2026-05-04,Yogyakarta,Shopee,Returned,<NA>,42900.00
5,SHP-0007,EMN-SKC-002,Emina Sun Battle SPF 50 PA++++ 30ml,Skincare,3,149700.00,2026-08-12,Surabaya,Shopee,Completed,<NA>,49900.00
6,SHP-0009,WND-BDY-001,Wonderly Body Mist Sweet Blossom 100ml,Fragrance,1,45900.00,2026-03-18,Bandung,Shopee,Completed,<NA>,45900.00
7,SHP-0010,MKO-MUP-002,Make Over Powerstay Matte Powder Foundation N20,Makeup,1,179000.00,2026-06-01,Surabaya,Shopee,Completed,<NA>,179000.00
8,SHP-0011,BDF-BDY-001,Biodef Body Wash Fresh Care 450ml,Body Care,1,35900.00,2026-04-16,Jakarta,Shopee,Completed,<NA>,35900.00
9,SHP-0012,WRD-MUP-001,Wardah Colorfit Velvet Matte Lip Mousse 03,Makeup,3,188700.00,2026-08-05,Yogyakarta,Shopee,Completed,<NA>,62900.00


Tersimpan: C:\Users\ADVAN\OneDrive - Universitas Teknologi Yogyakarta\Rinaldi\Ecommerce Sales\data\processed\clean
